# Session 7: joins, and how they make your numbers wrong

The fan-out section in the middle of this notebook is the thing to remember
in five years, when everything else here has faded.

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

## Why the data is in pieces

If `plays` held the artist details too, "Norway" would be written out 237
times for Fjord & Flint alone. Three problems, all really the same problem:

* **waste**: the same fact stored hundreds of times
* **drift**: fix a typo in one row and the other 236 still disagree
* **nowhere to put a fact**: an artist you have never listened to could not
  exist at all

So each fact lives in exactly one place. An artist's country is a fact about
the artist, so it belongs in `artists`. A play is a fact about one listen.

The cost of that tidiness is that most interesting questions now need both
tables at once. Putting them back together for the length of one query is
what a **join** is.

## Your first join

In [ ]:
run("""
SELECT p.played_at, a.artist_name, p.track_name, p.minutes_played
FROM plays AS p
JOIN artists AS a ON a.artist_id = p.artist_id
ORDER BY p.played_at
LIMIT 5
""")

Read the `ON` as the matching rule: "bring me the `artists` row whose
`artist_id` matches this play's `artist_id`".

`AS p` and `AS a` are **aliases**, short nicknames for the tables, so
`p.track_name` says which table a column came from.

## Ambiguous columns

Both tables have an `artist_id`, so asking for it unqualified is not a
question with one answer.

In [ ]:
try:
    run("SELECT artist_id, artist_name FROM plays JOIN artists "
        "ON artists.artist_id = plays.artist_id LIMIT 3")
except sqlite3.OperationalError as error:
    print("sqlite3.OperationalError:", error)

Get in the habit of aliasing every table and qualifying every column, even
when you do not have to. Queries grow, and the day you add a third table you
will be glad the first two were already labelled.

`USING` is the shorthand when the key has the same name on both sides:

In [ ]:
run("SELECT artist_id, artist_name, track_name FROM plays "
    "JOIN artists USING (artist_id) LIMIT 3")

## INNER versus LEFT JOIN

An inner join keeps only the rows that matched. A left join keeps **every
row of the first table**, filling the other side with `NULL` where there was
no match.

In [ ]:
run("""
SELECT COUNT(DISTINCT a.artist_id) AS artists_with_plays
FROM artists a JOIN plays p USING (artist_id)
""")

In [ ]:
run("""
SELECT COUNT(DISTINCT a.artist_id) AS all_artists
FROM artists a LEFT JOIN plays p USING (artist_id)
""")

38 against 40. Two artists have never been played, and the inner join hides
them completely.

**The question decides which join you want.** "How long did I listen to each
artist I played" is an inner join. "Which artists have I never played" needs
a left join, because the answer is made entirely of rows with no match.

In [ ]:
run("""
SELECT a.artist_name, a.genre, COUNT(p.play_id) AS plays
FROM artists a
LEFT JOIN plays p USING (artist_id)
GROUP BY a.artist_id
ORDER BY plays
LIMIT 4
""")

### Why `COUNT(p.play_id)` and not `COUNT(*)`

After a left join, an unmatched artist still produces one row, with every
`plays` column `NULL`. `COUNT(*)` counts that row and says **1**.
`COUNT(p.play_id)` skips NULLs and correctly says **0**.

Here is the wrong version. Change one word and "never played" becomes
"played once".

In [ ]:
run("""
SELECT a.artist_name, COUNT(*) AS plays_wrong
FROM artists a
LEFT JOIN plays p USING (artist_id)
GROUP BY a.artist_id
ORDER BY plays_wrong
LIMIT 4
""")

## Join, then group: the useful shape

In [ ]:
run("""
SELECT a.artist_name,
       a.country,
       COUNT(*)                        AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes
FROM plays p
JOIN artists a USING (artist_id)
GROUP BY a.artist_id
ORDER BY minutes DESC
LIMIT 5
""")

Look at the top rows. **Sara Lindqvist has the most plays. Glass Tram has
the most minutes.** Whichever you sort by changes who "my top artist" is,
and both are correct answers to different questions.

---

# Fan-out: the real lesson

Two queries. One number changed. Nothing was filtered, nothing was added.

In [ ]:
run("SELECT ROUND(SUM(minutes_played), 1) AS true_total FROM plays")

In [ ]:
run("""
SELECT ROUND(SUM(p.minutes_played), 1) AS total_with_awards_joined
FROM plays p
JOIN awards w ON w.artist_id = p.artist_id
""")

8,980.4 became 9,732.1. The total went up by 751 minutes because a table was
joined on.

Both queries ran without a word of complaint. If you had only ever run the
second one, you would have reported 9,732 and nobody in the room could have
told you otherwise.

## Count the rows, and it is obvious

In [ ]:
run("SELECT COUNT(*) AS rows_before FROM plays")

In [ ]:
run("""
SELECT COUNT(*) AS rows_after
FROM plays p JOIN awards w ON w.artist_id = p.artist_id
""")

2,183 became 2,371. Now one artist in detail: Fjord & Flint, `artist_id` 35.

In [ ]:
run("""
SELECT COUNT(*) AS plays, ROUND(SUM(minutes_played), 1) AS minutes
FROM plays WHERE artist_id = 35
""")

In [ ]:
run("SELECT award_name, year FROM awards WHERE artist_id = 35")

In [ ]:
run("""
SELECT COUNT(*) AS rows_now, ROUND(SUM(p.minutes_played), 1) AS minutes_now
FROM plays p JOIN awards w ON w.artist_id = p.artist_id
WHERE p.artist_id = 35
""")

237 plays, 3 awards, and after the join **711 rows**. Every play met every
award, so each play's minutes were counted three times.

The join has no idea the awards have nothing to do with the listening. It
only follows the rule you gave it: match on `artist_id`.

A **one-to-many** join multiplies rows, and every `SUM`, `COUNT` and `AVG`
downstream is wrong. That is fan-out.

**The habit worth building:** after any join, check the row count against
what it was before. If it went up and you did not expect it to, stop.

## A trap inside the trap

Everybody reaches for `DISTINCT` at this point.

In [ ]:
run("""
SELECT ROUND(SUM(DISTINCT p.minutes_played), 1) AS also_wrong
FROM plays p JOIN awards w ON w.artist_id = p.artist_id
WHERE p.artist_id = 35
""")

| Query | Answer | Right? |
|---|---|---|
| No join | `846.2` | Yes |
| Joined | `2538.5` | Three times too big |
| Joined with `SUM(DISTINCT)` | `680.6` | Too small now |

`SUM(DISTINCT)` adds up each **different value** once. Two separate plays
that both happened to last 4.65 minutes are two real plays, and it throws one
away. It fixed the duplication and destroyed the data.

The lesson is bigger than the keyword. **When a number looks wrong, do not
reach for the switch that makes it look right.** Work out what the rows
actually are first. Here, two wrongs sat either side of the correct answer,
and both looked like fixes.

## The real fix: aggregate first, then join

In [ ]:
run("""
WITH per_artist AS (
    SELECT artist_id,
           COUNT(*)                      AS plays,
           ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    GROUP BY artist_id
)
SELECT a.artist_name, s.plays, s.minutes, COUNT(w.award_id) AS awards
FROM per_artist s
JOIN artists a USING (artist_id)
LEFT JOIN awards w USING (artist_id)
GROUP BY a.artist_id
ORDER BY s.minutes DESC
LIMIT 3
""")

Fjord & Flint is back to **846.2**, which matches the standalone query
exactly. The fix is confirmed by an independent calculation, not by looking
sensible.

**Why it works:** the inner query reduces `plays` to one row per artist
before anything else touches it. After that the awards join cannot duplicate
a play, because there are no play rows left to duplicate.

## `WITH`, and why it beats a nested subquery

That `WITH per_artist AS (...)` is a **common table expression**, and
everybody says CTE. It is the same thing as putting the subquery inline, but
it is named and it sits at the top, so the query reads top to bottom like a
paragraph instead of inside-out.

Here is the same query written the nested way. Identical result, harder to
follow, and this one is short.

In [ ]:
run("""
SELECT a.artist_name, s.plays, s.minutes, COUNT(w.award_id) AS awards
FROM (
    SELECT artist_id, COUNT(*) AS plays,
           ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays GROUP BY artist_id
) AS s
JOIN artists a USING (artist_id)
LEFT JOIN awards w USING (artist_id)
GROUP BY a.artist_id
ORDER BY s.minutes DESC
LIMIT 3
""")

You can have several CTEs, separated by commas, and each can use the ones
above it. That is how a genuinely complicated question gets broken into
readable steps, and it is the single biggest readability win available in
SQL.

---

# When the numbers look wrong

| Symptom | Likely cause | Check |
|---|---|---|
| Totals larger than they should be | Fan-out from a one-to-many join | Count rows before and after |
| Rows have gone missing | An inner join where you needed a left join | Swap it and compare counts |
| Everything is zero or empty | The join keys do not actually match | Compare a few values by eye |
| Counts one too many per group | `COUNT(*)` after a left join | Use `COUNT(other_table.key)` |
| A percentage comes out as 0 | Whole-number division | Multiply by `100.0` first |

**Above all: know roughly what the answer should be before you run the
query.** 8,980 minutes over two years is about twelve minutes a day, which is
plausible. 9,732 would have passed unnoticed without a number to compare it
to.

An expectation is the cheapest bug detector you have.

In [ ]:
con.close()
print("closed")

## Next

`02-joins-exercise.ipynb`, where you break it on purpose and explain why.